In [ ]:
import pandas as pd

# Load the merged dataset
df = pd.read_csv('/Users/yaswanthkanagarla/Desktop/Master_Thesis/BD_code/Battery_Diagnostics_Thesis/scripts/merged_eis_capacity_state_IX_norm.csv')

state_map = {'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5, 'VI': 6, 'VII': 7, 'VIII': 8, 'IX': 9}
df['state'] = df['state'].map(state_map)

train_groups = [(25, 6), (25, 7), (25, 8), (25, 5), (25, 3), (25, 4), (35, 1)]
val_groups   = [(25, 2), (45, 2)]                 # validation cell(s)
test_groups  = [(25, 1), (35, 2), (45, 1)]         # final test cell(s)

def mask_from_groups(df, groups):
    groups = set(groups)
    return df.apply(lambda r: (r["T_C"], r["cell"]) in groups, axis=1)

train_mask = mask_from_groups(df, train_groups)
val_mask   = mask_from_groups(df, val_groups)
test_mask  = mask_from_groups(df, test_groups)

train_df = df[train_mask].copy()
val_df   = df[val_mask].copy()
test_df  = df[test_mask].copy()

print("Train:", train_df.shape, "Groups:", sorted(set(zip(train_df.T_C, train_df.cell))))
print("Val  :", val_df.shape,   "Groups:", sorted(set(zip(val_df.T_C, val_df.cell))))
print("Test :", test_df.shape,  "Groups:", sorted(set(zip(test_df.T_C, test_df.cell))))

Train: (1279, 31) Groups: [(25, 3), (25, 4), (25, 5), (25, 6), (25, 7), (25, 8), (35, 1)]
Val  : (491, 31) Groups: [(25, 2), (45, 2)]
Test : (966, 31) Groups: [(25, 1), (35, 2), (45, 1)]


In [30]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import matplotlib.pyplot as plt


def metrics(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred)))

         }
         
features = [
    "R_s_rel",
    "R_ct_rel",
    "Zmag_0.1Hz_rel",
    "Zmag_1000Hz_rel",
    "C_dl_log",
    "tail_slope",
    "R_sei_rel", "T_C" 

]  


X_train, y_train = train_df[features], train_df["SOH"]
X_val,   y_val   = val_df[features],   val_df["SOH"]
X_test,  y_test  = test_df[features],  test_df["SOH"]


In [31]:
from catboost import CatBoostRegressor

cat = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    verbose=0,
    random_seed=42
)

cat.fit(X_train, y_train)

print("Train:", metrics(y_train, cat.predict(X_train)))
print("Val  :", metrics(y_val,   cat.predict(X_val)))
print("Test :", metrics(y_test,  cat.predict(X_test)))

Train: {'R2': 0.9994704432643315, 'MAE': 0.003512537110013234, 'RMSE': 0.004539689727492659}
Val  : {'R2': 0.5135841369045819, 'MAE': 0.045121562429948456, 'RMSE': 0.049844265348701816}
Test : {'R2': 0.9011456654680281, 'MAE': 0.04627130283788786, 'RMSE': 0.05708711331423807}


In [24]:
import numpy as np

X_train_c = X_train.replace([np.inf, -np.inf], np.nan)
X_val_c   = X_val.replace([np.inf, -np.inf], np.nan)
X_test_c  = X_test.replace([np.inf, -np.inf], np.nan)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

rng = np.random.default_rng(42)

def sample_params():
    return {
        "depth": int(rng.choice([4, 5, 6, 7, 8, 9, 10])),
        "learning_rate": float(rng.choice([0.01, 0.02, 0.03, 0.05, 0.08, 0.1])),
        "l2_leaf_reg": float(rng.choice([1, 3, 5, 7, 10, 15])),
        "subsample": float(rng.choice([0.7, 0.8, 0.9, 1.0])),
        "colsample_bylevel": float(rng.choice([0.7, 0.8, 0.9, 1.0])),
        "random_strength": float(rng.choice([0, 0.5, 1, 2])),
        "bagging_temperature": float(rng.choice([0, 0.5, 1, 2])),
    }

n_iter = 30  
results = []
best_rmse = np.inf
best_model = None
best_params = None

for i in range(n_iter):
    p = sample_params()

    cat = CatBoostRegressor(
        iterations=8000,
        loss_function="RMSE",
        random_seed=42,
        verbose=0,
        **p
    )

    cat.fit(
        X_train_c, y_train,
        eval_set=(X_val_c, y_val),
        use_best_model=True,
        early_stopping_rounds=300
    )

    pred = cat.predict(X_val_c)
    rmse = float(np.sqrt(mean_squared_error(y_val, pred)))

    row = {**p, "best_iteration": int(cat.get_best_iteration()), "val_rmse": rmse}
    results.append(row)

    if rmse < best_rmse:
        best_rmse = rmse
        best_model = cat
        best_params = row

res_df = pd.DataFrame(results).sort_values("val_rmse")
print("Best params:", best_params)
print("Best Val RMSE:", best_rmse)
res_df.head(10)


Best params: {'depth': 4, 'learning_rate': 0.08, 'l2_leaf_reg': 7.0, 'subsample': 0.8, 'colsample_bylevel': 0.8, 'random_strength': 2.0, 'bagging_temperature': 0.0, 'best_iteration': 31, 'val_rmse': 0.041894343353079365}
Best Val RMSE: 0.041894343353079365


,depth,learning_rate,l2_leaf_reg,subsample,colsample_bylevel,random_strength,bagging_temperature,best_iteration,val_rmse
0,4,0.08,7.0,0.8,0.8,2.0,0.0,31,0.041894
21,5,0.08,1.0,1.0,0.8,2.0,0.5,28,0.042886
18,4,0.08,7.0,0.9,0.9,1.0,0.0,33,0.043355
24,6,0.01,1.0,0.7,1.0,1.0,1.0,231,0.043370
17,4,0.03,10.0,0.9,0.8,2.0,1.0,120,0.043965
25,7,0.08,1.0,1.0,0.9,2.0,0.0,29,0.044475
11,5,0.05,10.0,1.0,0.8,0.0,2.0,32,0.044769
16,7,0.03,7.0,0.9,0.7,0.0,0.0,56,0.045326
3,6,0.02,15.0,1.0,0.9,0.5,2.0,118,0.045531
4,7,0.03,5.0,0.7,0.7,1.0,2.0,85,0.045679


In [27]:
print("Train:", metrics(y_train, best_model.predict(X_train_c)))
print("Val  :", metrics(y_val,   best_model.predict(X_val_c)))
print("Test :", metrics(y_test,  best_model.predict(X_test_c)))


Train: {'R2': 0.9324689576101826, 'MAE': 0.03797166757966325, 'RMSE': 0.051265081225667766}
Val  : {'R2': 0.6563723464411788, 'MAE': 0.03668755362146786, 'RMSE': 0.041894343353079365}
Test : {'R2': 0.9030526390145481, 'MAE': 0.047286635580663414, 'RMSE': 0.05653380547846327}
